In [1]:
!pwd

/Users/codebase/Documents/codebase/courses/learn-flink/src/learn_flink


In [2]:
import os
from pyflink.table import EnvironmentSettings, TableEnvironment

base = "/Users/codebase/Documents/codebase/courses/learn-flink/src/learn_flink"
jars = [
    f"{base}/flink-sql-connector-kafka-4.0.0-2.0.jar",
    f"{base}/flink-sql-avro-confluent-registry-2.0.0.jar",  # needed for avro-confluent format
]
for j in jars:
    assert os.path.exists(j), f"JAR not found: {j}"

table_env = TableEnvironment.create(EnvironmentSettings.in_streaming_mode())
table_env.get_config().set("pipeline.jars", ";".join(f"file://{j}" for j in jars))

/Users/codebase/Documents/codebase/courses/learn-flink/.venv/lib/python3.8/site-packages/apache_beam/__init__.py:79: UserWarning: Python 3.8 reaches EOL in October 2024 and support will be removed from Apache Beam in version 2.61.0. See https://github.com/apache/beam/issues/31192 for more information.
  warnings.warn(


In [3]:
table_env.execute_sql(
    """
        CREATE TABLE readings (
          sensor_id   STRING,
          temperature DOUBLE,
          `timestamp` TIMESTAMP(3)
        ) WITH (
          'connector' = 'kafka',
          'topic' = 'thermostat_readings',
          'properties.bootstrap.servers' = 'localhost:9092',
          'properties.group.id' = 'python_kafka101_group_1',
          'scan.startup.mode' = 'earliest-offset',
          'format' = 'avro-confluent',
          'avro-confluent.url' = 'http://localhost:8081'
        );
    """
)

In [4]:
# Streaming query never ends. Read a bounded number of rows; `with` cancels the job cleanly.
# Don't interrupt the kernel while this runs - it can leave PyFlink's planner in a broken state
# (NullPointerException: metadataHandlerProvider on next run). If that happens: restart kernel.
# from itertools import islice

# with table_env.execute_sql("SELECT * FROM readings WHERE sensor_id = 'thermostat-1'").collect() as rows:
#     for row in islice(rows, 10):
#         print(row)

In [5]:
from itertools import islice

def peek(sql, n=10):
    with table_env.execute_sql(sql).collect() as rows:
        for row in islice(rows, n):
            print(row)

peek("SELECT * FROM readings")
peek("SELECT sensor_id, temperature FROM readings WHERE temperature > 25", 5)

<Row('thermostat-3', 27.0, datetime.datetime(2026, 11, 5, 10, 50, 42, 969000))>
<Row('thermostat-3', 18.7, datetime.datetime(2026, 11, 5, 10, 50, 43, 633000))>
<Row('thermostat-2', 18.5, datetime.datetime(2026, 11, 5, 10, 50, 44, 138000))>
<Row('thermostat-3', 19.7, datetime.datetime(2026, 11, 5, 10, 50, 44, 639000))>
<Row('thermostat-2', 28.2, datetime.datetime(2026, 11, 5, 10, 50, 45, 144000))>
<Row('thermostat-3', 16.2, datetime.datetime(2026, 11, 5, 11, 15, 49, 574000))>
<Row('thermostat-3', 19.1, datetime.datetime(2026, 11, 5, 11, 15, 49, 699000))>
<Row('thermostat-1', 22.4, datetime.datetime(2026, 11, 5, 11, 15, 49, 805000))>
<Row('thermostat-1', 26.9, datetime.datetime(2026, 11, 5, 11, 53, 13, 123000))>
<Row('thermostat-1', 22.3, datetime.datetime(2026, 11, 5, 11, 53, 13, 657000))>
<Row('thermostat-3', 27.0)>
<Row('thermostat-2', 28.2)>
<Row('thermostat-1', 26.9)>
<Row('thermostat-3', 26.1)>
<Row('thermostat-3', 26.7)>


In [6]:
peek(
    """
    SELECT * FROM readings WHERE sensor_id = 'thermostat-1'
    """
)

<Row('thermostat-1', 22.4, datetime.datetime(2026, 11, 5, 11, 15, 49, 805000))>
<Row('thermostat-1', 26.9, datetime.datetime(2026, 11, 5, 11, 53, 13, 123000))>
<Row('thermostat-1', 22.3, datetime.datetime(2026, 11, 5, 11, 53, 13, 657000))>
<Row('thermostat-1', 25.6, datetime.datetime(2026, 11, 5, 11, 53, 15, 171000))>
<Row('thermostat-1', 26.6, datetime.datetime(2026, 11, 5, 11, 54, 59, 803000))>
<Row('thermostat-1', 20.2, datetime.datetime(2026, 11, 5, 12, 2, 19, 846000))>
<Row('thermostat-1', 15.8, datetime.datetime(2026, 11, 5, 12, 2, 30, 801000))>
<Row('thermostat-1', 27.3, datetime.datetime(2026, 11, 5, 12, 2, 31, 306000))>
<Row('thermostat-1', 17.5, datetime.datetime(2026, 11, 5, 12, 2, 47, 275000))>
<Row('thermostat-1', 26.8, datetime.datetime(2026, 11, 5, 12, 2, 47, 778000))>


In [7]:
peek(
    """
    SELECT * FROM readings WHERE sensor_id = 'thermostat-1'
    """
)

<Row('thermostat-1', 22.4, datetime.datetime(2026, 11, 5, 11, 15, 49, 805000))>
<Row('thermostat-1', 26.9, datetime.datetime(2026, 11, 5, 11, 53, 13, 123000))>
<Row('thermostat-1', 22.3, datetime.datetime(2026, 11, 5, 11, 53, 13, 657000))>
<Row('thermostat-1', 25.6, datetime.datetime(2026, 11, 5, 11, 53, 15, 171000))>
<Row('thermostat-1', 26.6, datetime.datetime(2026, 11, 5, 11, 54, 59, 803000))>
<Row('thermostat-1', 20.2, datetime.datetime(2026, 11, 5, 12, 2, 19, 846000))>
<Row('thermostat-1', 15.8, datetime.datetime(2026, 11, 5, 12, 2, 30, 801000))>
<Row('thermostat-1', 27.3, datetime.datetime(2026, 11, 5, 12, 2, 31, 306000))>
<Row('thermostat-1', 17.5, datetime.datetime(2026, 11, 5, 12, 2, 47, 275000))>
<Row('thermostat-1', 26.8, datetime.datetime(2026, 11, 5, 12, 2, 47, 778000))>


In [8]:
peek(
    """
    SELECT * FROM readings WHERE sensor_id = 'thermostat-1'
    """
)

<Row('thermostat-1', 22.4, datetime.datetime(2026, 11, 5, 11, 15, 49, 805000))>
<Row('thermostat-1', 26.9, datetime.datetime(2026, 11, 5, 11, 53, 13, 123000))>
<Row('thermostat-1', 22.3, datetime.datetime(2026, 11, 5, 11, 53, 13, 657000))>
<Row('thermostat-1', 25.6, datetime.datetime(2026, 11, 5, 11, 53, 15, 171000))>
<Row('thermostat-1', 26.6, datetime.datetime(2026, 11, 5, 11, 54, 59, 803000))>
<Row('thermostat-1', 20.2, datetime.datetime(2026, 11, 5, 12, 2, 19, 846000))>
<Row('thermostat-1', 15.8, datetime.datetime(2026, 11, 5, 12, 2, 30, 801000))>
<Row('thermostat-1', 27.3, datetime.datetime(2026, 11, 5, 12, 2, 31, 306000))>
<Row('thermostat-1', 17.5, datetime.datetime(2026, 11, 5, 12, 2, 47, 275000))>
<Row('thermostat-1', 26.8, datetime.datetime(2026, 11, 5, 12, 2, 47, 778000))>


In [9]:
peek(
    """
    SELECT * FROM readings WHERE sensor_id = 'thermostat-1'
    """
)

<Row('thermostat-1', 22.4, datetime.datetime(2026, 11, 5, 11, 15, 49, 805000))>
<Row('thermostat-1', 26.9, datetime.datetime(2026, 11, 5, 11, 53, 13, 123000))>
<Row('thermostat-1', 22.3, datetime.datetime(2026, 11, 5, 11, 53, 13, 657000))>
<Row('thermostat-1', 25.6, datetime.datetime(2026, 11, 5, 11, 53, 15, 171000))>
<Row('thermostat-1', 26.6, datetime.datetime(2026, 11, 5, 11, 54, 59, 803000))>
<Row('thermostat-1', 20.2, datetime.datetime(2026, 11, 5, 12, 2, 19, 846000))>
<Row('thermostat-1', 15.8, datetime.datetime(2026, 11, 5, 12, 2, 30, 801000))>
<Row('thermostat-1', 27.3, datetime.datetime(2026, 11, 5, 12, 2, 31, 306000))>
<Row('thermostat-1', 17.5, datetime.datetime(2026, 11, 5, 12, 2, 47, 275000))>
<Row('thermostat-1', 26.8, datetime.datetime(2026, 11, 5, 12, 2, 47, 778000))>
